# Filters and FieldMaps

Standalone exploration of `FilterHandler`, `Expr` composability, and `FieldMap` bidirectional translation.

| # | Topic |
|---|---|
| 1 | FieldMap — DB to semantic name translation |
| 2 | FieldMap — filter translation, strict mode, DataFrame renaming |
| 3 | FilterHandler — backend-neutral filter compilation |
| 4 | Expr tree — And, Or, Not composability |
| 5 | FilterHandler — pushdown vs residual split for parquet |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import pandas as pd

from boti_data import FieldMap, FilterHandler, And, Or, Not, Expr, TrueExpr

## 1. FieldMap — bidirectional name translation

FieldMap maps **DB column names** to **semantic names**. The convention is `{db_column: semantic_name}`.

In [2]:
mapping = {
    "id_tipo_producto": "product_type_id",
    "nb_razon_social": "company_name",
    "fc_fecha_creacion": "created_at",
    "importe": "amount",
}
fm = FieldMap(mapping)

print(f"FieldMap: {fm}")
print(f"  DB columns:       {fm.db_columns}")
print(f"  Semantic names:   {fm.semantic_names}")
print(f"  to_db('company_name'):  {fm.to_db('company_name')}")
print(f"  to_semantic('importe'): {fm.to_semantic('importe')}")
print(f"  to_db('unknown'):       {fm.to_db('unknown')}  (pass-through)")

FieldMap: FieldMap(4 columns)
  DB columns:       ['id_tipo_producto', 'nb_razon_social', 'fc_fecha_creacion', 'importe']
  Semantic names:   ['product_type_id', 'company_name', 'created_at', 'amount']
  to_db('company_name'):  nb_razon_social
  to_semantic('importe'): amount
  to_db('unknown'):       unknown  (pass-through)


### 1b. Strict mode rejects unknown keys

When `strict=True`, `to_db()` raises `KeyError` for unmapped semantic names — useful as a security boundary.

In [3]:
strict_fm = FieldMap(mapping, strict=True)
try:
    strict_fm.to_db("malicious_field")
except KeyError as e:
    print(f"Strict mode blocked: {e}")

Strict mode blocked: "Unknown semantic field 'malicious_field': not found in FieldMap. Pass strict=False or add the field to the mapping."


## 2. FieldMap — filter translation and DataFrame renaming

Filters expressed with semantic keys are translated to DB column names before hitting the database.

In [4]:
semantic_filters = {
    "product_type_id__in": [1, 2, 3],
    "company_name__like": "%Corp",
    "amount__gte": 1000,
}
db_filters = fm.translate_filters_to_db(semantic_filters)
print("Semantic filters:", semantic_filters)
print("DB filters:      ", db_filters)

# Nested boolean operators are also translated recursively
nested = {
    "$or": [
        {"product_type_id": 1},
        {"company_name__like": "%LLC"},
    ]
}
print("Nested semantic:", nested)
print("Nested DB:      ", fm.translate_filters_to_db(nested))

Semantic filters: {'product_type_id__in': [1, 2, 3], 'company_name__like': '%Corp', 'amount__gte': 1000}
DB filters:       {'id_tipo_producto__in': [1, 2, 3], 'nb_razon_social__like': '%Corp', 'importe__gte': 1000}
Nested semantic: {'$or': [{'product_type_id': 1}, {'company_name__like': '%LLC'}]}
Nested DB:       {'$or': [{'id_tipo_producto': 1}, {'nb_razon_social__like': '%LLC'}]}


In [5]:
# DataFrame renaming: DB -> semantic
df = pd.DataFrame({
    "id_tipo_producto": [1, 2],
    "nb_razon_social": ["Acme", "Beta"],
    "importe": [100.0, 200.0],
})
semantic_df = fm.rename_dataframe(df)
print("DB columns:     ", list(df.columns))
print("Semantic columns:", list(semantic_df.columns))
# Only mapped columns are renamed; unmapped ones pass through unchanged

DB columns:      ['id_tipo_producto', 'nb_razon_social', 'importe']
Semantic columns: ['product_type_id', 'company_name', 'amount']


## 3. FilterHandler — backend-neutral filter compilation

`FilterHandler` compiles filter dicts into backend-specific expressions (SQLAlchemy, Dask, Arrow).

In [6]:
handler = FilterHandler(backend="dask")

filters = {
    "amount__gte": 100,
    "status__in": ["active", "pending"],
    "name__like": "%Corp",
}

# Compile to an Expr tree
expr = handler.compile_filters(filters)
print(f"Compiled Expr: {expr}")
print(f"  Is trivial? {expr.is_trivial()}")

# Build a mask function for Dask DataFrames
mask_fn = handler.build_mask_fn(filters)
print(f"  mask_fn: {mask_fn}")

Compiled Expr: And(left=And(left=ColOp(field='amount', casting=None, op='gte', value=100, handler=<boti_data.filters.handler.FilterHandler object at 0x15314da90>), right=ColOp(field='status', casting=None, op='in', value=['active', 'pending'], handler=<boti_data.filters.handler.FilterHandler object at 0x15314da90>)), right=ColOp(field='name', casting=None, op='like', value='%Corp', handler=<boti_data.filters.handler.FilterHandler object at 0x15314da90>))
  Is trivial? False
  mask_fn: <function FilterHandler.build_mask_fn.<locals>._mask at 0x107e6a840>


In [7]:
# Suggest IN clause chunking for SQL backends
chunked = handler.suggest_sql_in_chunking(filters, chunk_size=2, max_concurrency=4)
print(f"IN chunking suggestion: {chunked}")

IN chunking suggestion: None


## 4. Expr tree — And, Or, Not composability

Expression trees can be built programmatically for complex filter logic.

In [8]:
# Build: (status == 'active' OR amount > 500) AND NOT (name == 'test')
from boti_data.filters import ColOp

active_expr = ColOp(field="status", casting=None, op="exact", value="active", handler=handler)
amount_expr = ColOp(field="amount", casting=None, op="gt", value=500, handler=handler)
name_expr   = ColOp(field="name", casting=None, op="exact", value="test", handler=handler)

composite = And(
    Or(active_expr, amount_expr),
    Not(name_expr),
)
print(f"Composite: {composite}")
print(f"  Is trivial? {composite.is_trivial()}")

# Empty / always-true expressions
print(f"TrueExpr always trivial: {TrueExpr().is_trivial()}")

TypeError: ColOp.__init__() missing 1 required positional argument: 'casting'

## 5. FilterHandler — pushdown vs residual split

For parquet loads, the handler splits filters into pushdown candidates (parquet predicate pushdown) and residual filters (applied in-memory).

In [ ]:
handler_arrow = FilterHandler(backend="arrow")

mixed_filters = {
    "amount__gte": 100,
    "status__in": ["active", "pending"],
    "tags__contains": "premium",      # residual — not pushdownable
    "name__date__exact": "2024-01-01", # date casting -> residual
}

pushdown, residual = handler_arrow.split_pushdown_and_residual(mixed_filters)
print(f"Pushdown filters:   {pushdown}")
print(f"Residual filters:   {residual}")

# Convert pushdown candidates to parquet-style tuple format
parquet_filters = handler_arrow.to_parquet_filters(mixed_filters)
print(f"Parquet filters:    {parquet_filters}")

### Summary

- **FieldMap** provides bidirectional DB↔semantic name translation, strict-mode key rejection, filter dict translation, and DataFrame renaming.
- **FilterHandler** compiles filter dicts into backend-specific expressions, splits pushdown vs residual for parquet, and can chunk large `IN` clauses.
- **Expr** trees (And, Or, Not, ColOp, TrueExpr) can be composed programmatically for complex filter logic.